[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaronJenish/zawdwadw/blob/main/Experiment_11_CNN_for_Image_Classification.ipynb)


# Experiment 11: CNN for Image Classification

CSA47 – Deep Learning. Run the setup cell first, then execute the experiment cells in order.


In [ ]:
# Common setup – run this cell first
import os, time, warnings, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_absolute_error, mean_squared_error)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
print("Setup complete")

## Experiment 11 – CNN for Image Classification

In [ ]:
# CIFAR-10 substitute dataset
import tensorflow as tf, time
(xtrain,ytrain),(xtest,ytest)=tf.keras.datasets.cifar10.load_data()
xtrain=xtrain.astype("float32")/255; xtest=xtest.astype("float32")/255
ytrain=ytrain.ravel(); ytest=ytest.ravel()
# Keep a subset for practical Colab runtime; increase sizes for full training
xtr,ytr=xtrain[:20000],ytrain[:20000]; xv,yv=xtrain[20000:24000],ytrain[20000:24000]
cnn=tf.keras.Sequential([tf.keras.layers.Input((32,32,3)),
 tf.keras.layers.Conv2D(32,3,activation="relu",name="conv1"),tf.keras.layers.MaxPooling2D(),
 tf.keras.layers.Conv2D(64,3,activation="relu"),tf.keras.layers.MaxPooling2D(),
 tf.keras.layers.Flatten(),tf.keras.layers.Dense(64,activation="relu"),tf.keras.layers.Dense(10,activation="softmax")])
cnn.compile("adam","sparse_categorical_crossentropy",metrics=["accuracy"])
hist=cnn.fit(xtr,ytr,validation_data=(xv,yv),epochs=5,batch_size=64)
loss,acc=cnn.evaluate(xtest[:3000],ytest[:3000],verbose=0); print("Test accuracy:",acc)
plt.plot(hist.history["accuracy"],label="train"); plt.plot(hist.history["val_accuracy"],label="validation"); plt.legend(); plt.show()
pred=cnn.predict(xtest[:1000],verbose=0).argmax(1)
print("Confusion matrix:\n",confusion_matrix(ytest[:1000],pred))
# Misclassified samples
bad=np.where(pred!=ytest[:1000])[0][:8]
fig,axs=plt.subplots(1,max(1,len(bad)),figsize=(14,2))
for ax,i in zip(np.atleast_1d(axs),bad): ax.imshow(xtest[i]); ax.set_title(f"T:{ytest[i]} P:{pred[i]}"); ax.axis("off")
plt.show()
# First convolution feature maps
feature_model=tf.keras.Model(cnn.input,cnn.get_layer("conv1").output)
fm=feature_model.predict(xtest[:1],verbose=0)[0]
fig,axs=plt.subplots(1,8,figsize=(12,2))
for i,ax in enumerate(axs): ax.imshow(fm[:,:,i],cmap="viridis"); ax.axis("off")
plt.show()